# HMDA 2024: xuất bảng sạch (clean core) ra Parquet và đối chiếu từng dòng giữa Pandas và Spark

Hai notebook chính đều dựng bảng sạch rồi dùng ngay trong phiên chạy; Spark ghi nó ra Parquet ở thư mục tạm.
Notebook này giữ bảng đó lại làm một sản phẩm dữ liệu có thể tải về:
1. Chạy đúng phần 1–2 của `pandas_pipeline.py` (đọc, lọc mẫu core, tạo biến) và giữ bảng Pandas.
2. Chạy đúng phần 1–2 của `spark_pipeline.py`; Spark ghi bảng sạch ra Parquet, chia thư mục theo `state_code`.
3. Đọc lại Parquet do Spark ghi và so **từng dòng, từng cột** với bảng Pandas (khóa `row_id`).
4. Chép bảng sạch vào output (`clean_core/`), kèm từ điển cột và manifest số dòng theo bang.

Trên Kaggle: các ô phần 1–2 của hai pipeline được chép ngay sau ô này. Ngoài Kaggle:
`HMDA_INPUT=... python export_clean_core.py` (script tự chạy các ô đó).

In [1]:
import os
import sys

IN_NOTEBOOK = "ipykernel" in sys.modules
_HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."


def _cells_until(fname, stop_marker):
    """Các ô code của một pipeline, từ đầu đến trước tiêu đề `stop_marker`."""
    src = open(os.path.join(_HERE, fname), encoding="utf-8").read()
    src = src[:src.index(stop_marker)]
    return [c_ for c_ in src.split("# %%") if c_.strip() and not c_.startswith(" [markdown]")]


if not IN_NOTEBOOK:
    for _c in _cells_until("pandas_pipeline.py", "# ## 3. Kiểm tra với expected.yaml"):
        exec(compile(_c, "pandas_pipeline.py", "exec"))

---
# Phần 1–2 của `pandas_pipeline.py`

# HMDA 2024: pipeline Pandas (rules.yaml 1.1)

Notebook này thay cho notebook Pandas cũ, viết lại theo `rules.yaml` 1.1 và bổ sung các phần còn thiếu.

**Chuẩn bị trên Kaggle**
1. Input gồm dataset `hmda-2024-one-year-raw` (`hmda_2024_raw.parquet`) và `hmda-config` (`rules.yaml` bản 1.1, `expected.yaml`).
   Transmittal Sheet lấy từ dataset `hmda-2024-ffiec-files` (Kaggle nhập từ link FFIEC); thiếu file này thì notebook thử tải khi có Internet.
2. Settings: Accelerator = None (CPU), RAM 30 GB. Chạy *Run All*.
3. Kết quả ghi vào `/kaggle/working/results/pandas/` và `/kaggle/working/figures/pandas/`.
   Các file `eq_*.csv` và `key_numbers.json` dùng cho script so sánh với PySpark.

**Thứ tự**: (0) cài đặt, (1) đọc và lọc, (2) tạo biến, (3) kiểm tra số kỳ vọng, (4) SQ1, (5) SQ2,
(6) giá trị kỳ vọng p_ext, (7) SQ3, (8) SQ4, (9) mô hình hai chiều và HHI, (10) đầu ra tương đương,
(11) mô hình dự đoán, (12) tổng kết.

In [2]:
# ===== 0. Cài đặt =====
import os, glob, json, time, shutil, gc, hashlib, math
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import yaml
import matplotlib

try:
    get_ipython  # noqa: F821
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

T_START = time.time()
RUN_DUPLICATES = True      # đếm dòng trùng trên 99 cột (tốn vài phút trên dữ liệu thật)
RUN_PREDICTION = True      # mô hình dự đoán (mục 11)


def find_file(pattern, required=True):
    """Tìm file ở mọi dataset/notebook đã Add Input (hoặc thư mục HMDA_INPUT khi chạy ngoài Kaggle)."""
    roots = [os.environ["HMDA_INPUT"]] if os.environ.get("HMDA_INPUT") else ["/kaggle/input"]
    hits = sorted(h for r_ in roots for h in glob.glob(os.path.join(r_, "**", pattern), recursive=True))
    assert hits or not required, f"Không thấy {pattern} trong {roots}: kiểm tra Add Input"
    return hits[0] if hits else None


RAW_SRC = find_file("hmda_2024_raw.parquet")
RULES_PATH = find_file("rules.yaml")
EXP_PATH = find_file("expected.yaml", required=False)
TS_PATH = find_file("*public_ts*.csv", required=False) or find_file("*public_ts*.zip", required=False)
INP = os.path.dirname(RAW_SRC)
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
RES, FIG = f"{OUTROOT}/results/pandas", f"{OUTROOT}/figures/pandas"
WORK = os.environ.get("HMDA_WORK", "/tmp/hmda")
for d in (RES, FIG, WORK):
    os.makedirs(d, exist_ok=True)

R = yaml.safe_load(open(RULES_PATH, encoding="utf-8"))
assert float(R["version"]) >= 1.1, f"Cần rules.yaml 1.1 trở lên, đang là {R.get('version')}"
if TS_PATH is None and R["source"]["transmittal_sheet"].get("url"):   # chưa có TS trong Input: thử tải từ FFIEC (cần Internet)
    try:
        import urllib.request
        os.makedirs(os.environ.get("HMDA_WORK", "/tmp/hmda"), exist_ok=True)
        TS_PATH = os.path.join(os.environ.get("HMDA_WORK", "/tmp/hmda"), "ts_one_year.zip")
        urllib.request.urlretrieve(R["source"]["transmittal_sheet"]["url"], TS_PATH)
    except Exception as e_:
        print("không tải được Transmittal Sheet:", repr(e_)[:100]); TS_PATH = None
EXP = yaml.safe_load(open(EXP_PATH, encoding="utf-8")) if EXP_PATH else {}

RAW = f"{WORK}/hmda_2024_raw.parquet"
if not os.path.exists(RAW) or os.path.getsize(RAW) != os.path.getsize(RAW_SRC):
    t = time.time(); shutil.copy(RAW_SRC, RAW)
    print(f"đã chép Parquet sang {WORK} ({time.time() - t:.0f}s)")

V, TH, G = R["variables"], R["thresholds"], R["geography"]
LT_NAME = {k: v for k, v in R["codes"]["loan_type"].items()}
LP_NAME = {k: v for k, v in R["codes"]["loan_purpose"].items()}
UNIVERSE = set(G["universe_states"])
KEY = {"rules_version": str(R["version"]), "engine": "pandas"}   # số chính, so với PySpark
import platform, sklearn
KEY["env_pandas_notebook"] = (f"python {platform.python_version()}; numpy {np.__version__}; pandas {pd.__version__}; "
                              f"pyarrow {pa.__version__}; scikit-learn {sklearn.__version__}; matplotlib {matplotlib.__version__}")
print(KEY["env_pandas_notebook"])
CHECKS = []


def exp_value(path):
    node = EXP
    for p in path.split("."):
        if not isinstance(node, dict) or p not in node:
            return None, None, None
        node = node[p]
    return (node.get("value"), node.get("status"), node.get("tol", 1e-9)) if isinstance(node, dict) else (node, None, 1e-9)


def check(label, value, path=None):
    ev, st, tol = exp_value(path) if path else (None, None, None)
    # tol là sai số TƯƠNG ĐỐI (như ghi trong expected.yaml); kỳ vọng bằng 0 thì so tuyệt đối
    ok = None if (ev is None or value is None or st == "pending") else (
        abs(float(value) - float(ev)) <= tol * (abs(float(ev)) if float(ev) != 0 else 1.0))
    CHECKS.append({"check": label, "value": value, "expected": ev, "expected_status": st, "match": ok})
    flag = "" if ok is None else ("  OK" if ok else "  <-- LỆCH")
    print(f"  {label:<44} {value!s:>16}   kỳ vọng {ev!s:>12} ({st}){flag}")


def save(df, name, index=False):
    df.to_csv(f"{RES}/{name}.csv", index=index)


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.divide(k, n, out=np.full_like(k, np.nan), where=n > 0)
    den = 1 + z ** 2 / n
    ctr = (p + z ** 2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return ctr - half, ctr + half


def ntile(values, keys, k=5):
    """Giống Spark ntile(k) trên ORDER BY (value, key): k nhóm, các nhóm đầu nhận phần dư."""
    order = np.lexsort((np.asarray(keys), np.asarray(values)))
    n = len(order); base, rem = divmod(n, k)
    sizes = [base + (1 if i < rem else 0) for i in range(k)]
    lab = np.repeat(np.arange(1, k + 1), sizes)
    out = np.empty(n, dtype=int); out[order] = lab
    return out


def spearman(x, y):
    x, y = pd.Series(x).reset_index(drop=True), pd.Series(y).reset_index(drop=True)
    m = x.notna() & y.notna()
    return float(x[m].rank(method="average").corr(y[m].rank(method="average")))


def occupancy_table(occ):
    """Tỷ lệ hồ sơ không phải nơi ở chính theo loan_type, trên các hồ sơ đã qua mọi bước lọc trước bước nơi ở chính."""
    pr_ = next(f["equals"] for f in R["samples"]["core_filters"] if f["step"] == "principal_residence")
    occ = occ.assign(loan_type=occ["loan_type"].astype(str), occupancy_type=occ["occupancy_type"].astype(str))
    t = occ.groupby("loan_type")["n"].sum().rename("n_before_filter").to_frame()
    t["n_non_principal"] = (occ[occ["occupancy_type"] != pr_].groupby("loan_type")["n"].sum()
                            .reindex(t.index).fillna(0).astype("int64"))
    t["share_non_principal"] = t["n_non_principal"] / t["n_before_filter"]
    t.insert(0, "type", [LT_NAME.get(k, k) for k in t.index])
    return t.reset_index().sort_values("loan_type").reset_index(drop=True)


def show():
    if IN_NOTEBOOK:
        plt.show()
    plt.close("all")


print("input :", INP); print("output:", RES); print("rules :", R["version"], "| expected.yaml:", bool(EXP))

không tải được Transmittal Sheet: URLError(gaierror(-3, 'Temporary failure in name resolution'))
đã chép Parquet sang /tmp/hmda (2s)
python 3.13.18; numpy 2.1.3; pandas 2.3.3; pyarrow 23.0.1; scikit-learn 1.6.1; matplotlib 3.10.0
input : /kaggle/input/notebooks/study665/hmda-v2-convert
output: /kaggle/working/results/pandas
rules : 1.1 | expected.yaml: True


## 1. Đọc dữ liệu thô và lọc mẫu core
Một lượt đọc theo khối (200 nghìn dòng) bằng PyArrow, làm đồng thời:
số dòng, cơ cấu `action_taken` trên mọi bản ghi, bảng mã đặc biệt của 10 trường (rules mục 16),
đếm dòng trùng (hash 64-bit trên 99 cột, bỏ `row_id`), và lọc 7 bước của mẫu core (rules mục 4).

In [3]:
t0 = time.time()
pf = pq.ParquetFile(RAW)
ALLCOLS = [c for c in pf.schema_arrow.names if c != "row_id"]
FILT = R["samples"]["core_filters"]
DQ = R["data_quality"]["special_code_table"]
KEEP = ["row_id", "lei", "state_code", "county_code", "derived_msa_md", "action_taken", "loan_type",
        "loan_purpose", "loan_amount", "income", "debt_to_income_ratio", "combined_loan_to_value_ratio",
        "tract_to_msa_income_percentage", "denial_reason_1", "denial_reason_2", "denial_reason_3",
        "denial_reason_4", "rate_spread", "loan_term"]
NEED = sorted(set(KEEP) | {f["column"] for f in FILT} | set(DQ["fields"]))
READ = ALLCOLS + ["row_id"]          # đọc đủ 99 cột: profiling toàn bộ trường (rules mục 16)
FP = R["data_quality"]["field_profile"]

n_all = 0
act_counts = {}
sc_counts = {(f, c): 0 for f in DQ["fields"] for c in DQ["codes"]}
fp_counts = {(f, c): 0 for f in ALLCOLS for c in FP["count_codes"] + ["numeric"]}
step_counts = {f["step"]: 0 for f in FILT}
open_end_exempt_removed = 0
occ_counts = {}                        # (loan_type, occupancy_type) ngay trước bước lọc nơi ở chính
dup_hashes = []
parts = []
for batch in pf.iter_batches(batch_size=200_000, columns=READ):
    tb = pa.Table.from_batches([batch])
    n_all += tb.num_rows
    for v, cnt in zip(*[x.to_pylist() for x in pc.value_counts(tb["action_taken"]).flatten()]):
        act_counts[v] = act_counts.get(v, 0) + cnt
    for f in DQ["fields"]:
        col = tb[f]
        for code in DQ["codes"]:
            sc_counts[(f, code)] += int(pc.sum(pc.equal(col, code)).as_py() or 0)
    for f in ALLCOLS:
        col = tb[f]
        for code in FP["count_codes"]:
            fp_counts[(f, code)] += int(pc.sum(pc.equal(col, code)).as_py() or 0)
        fp_counts[(f, "numeric")] += int(pc.sum(pc.match_substring_regex(col, FP["numeric_pattern"])).as_py() or 0)
    if RUN_DUPLICATES:
        h = pd.util.hash_pandas_object(tb.select(ALLCOLS).to_pandas(), index=False).to_numpy()
        dup_hashes.append(h)
    for f in FILT:
        col = tb[f["column"]]
        if f["step"] == "closed_end":
            open_end_exempt_removed += int(pc.sum(pc.equal(col, "1111")).as_py() or 0)
        if f["step"] == "principal_residence":
            for k_, n_ in tb.select(["loan_type", f["column"]]).to_pandas().value_counts().items():
                occ_counts[k_] = occ_counts.get(k_, 0) + int(n_)
        mask = pc.is_in(col, pa.array(f["in"])) if "in" in f else pc.equal(col, f["equals"])
        tb = tb.filter(mask)
        step_counts[f["step"]] += tb.num_rows
    parts.append(tb.select(KEEP))

core_tab = pa.concat_tables(parts); del parts
c = core_tab.to_pandas(strings_to_categorical=True); del core_tab
c["row_id"] = c["row_id"].astype("int64")
n_dup = 0
if RUN_DUPLICATES:
    hh = np.concatenate(dup_hashes); del dup_hashes
    _, cnt = np.unique(hh, return_counts=True)
    n_dup = int((cnt - 1).sum()); del hh
gc.collect()
print(f"đọc {n_all:,} dòng, core {len(c):,} dòng | {time.time() - t0:.0f}s")

đọc 12,259,199 dòng, core 5,775,325 dòng | 273s


## 2. Tạo biến (rules mục 6)
Mọi chuyển đổi số đi qua danh mục (category) nên chỉ chuyển mỗi giá trị khác nhau một lần.
Mã `NA`, `Exempt`, chuỗi rỗng thành NaN; chuỗi lạ khác cũng thành NaN nhưng được đếm và in ra.

In [4]:
t0 = time.time()
NUM_SPECIAL = ["NA", "Exempt", ""]
STRANGE = {}


def to_num(s, name):
    if not isinstance(s.dtype, pd.CategoricalDtype):
        s = s.astype("category")
    cats = pd.Series(s.cat.categories.astype(str))
    vals = pd.to_numeric(cats.where(~cats.isin(NUM_SPECIAL)), errors="coerce").to_numpy(float)
    bad = cats[~cats.isin(NUM_SPECIAL) & np.isnan(vals)]
    if len(bad):
        STRANGE[name] = list(bad[:5])
    codes = s.cat.codes.to_numpy()
    return np.where(codes >= 0, vals[np.clip(codes, 0, None)], np.nan)


def as_str(s):
    return s.astype(str)


def bands(x, spec, cuts=None):
    lo = [b["lo"] for b in spec]; hi = [b["hi"] for b in spec]
    rs = lambda v: cuts[v] if isinstance(v, str) else float(v)
    edges = [rs(lo[0])] + [rs(h) for h in hi]
    return pd.cut(x, bins=edges, labels=[b["label"] for b in spec], right=True)


for col in ["action_taken", "loan_type", "loan_purpose"]:
    c[col] = c[col].astype(str).astype("category")
act = c["action_taken"].astype(str)
c["decisioned"] = act.isin(R["samples"]["decisioned"]["action_taken_in"]).to_numpy()
c["denied"] = (act == "3").to_numpy()
c["orig"] = (act == "1").to_numpy()

# DTI
D = V["debt_to_income_ratio"]
cats = pd.Series(c["debt_to_income_ratio"].cat.categories.astype(str))
norm = cats.str.replace("%", "", regex=False).str.replace(" ", "", regex=False)
dmap = norm.map({str(k): v for k, v in D["ordinal_map"].items()}).to_numpy(float)
codes = c["debt_to_income_ratio"].cat.codes.to_numpy()
c["dti_ordinal"] = np.where(codes >= 0, dmap[np.clip(codes, 0, None)], np.nan)
c["dti_band"] = bands(c["dti_ordinal"], D["bands"])

# CLTV
L = V["combined_loan_to_value_ratio"]
c["cltv"] = to_num(c["combined_loan_to_value_ratio"], "cltv")
c["cltv_valid"] = (c["cltv"] > L["valid_range"]["gt"]) & (c["cltv"] <= L["valid_range"]["le"])
c["cltv_band"] = bands(c["cltv"].where(c["cltv_valid"]), L["bands"])

# Income, loan amount, LTI
c["income_num"] = to_num(c["income"], "income")
c["income_is_na"] = (c["income"].astype(str) == "NA").to_numpy()
c["loan_amount_num"] = to_num(c["loan_amount"], "loan_amount")
inc_ok = c["income_num"] > 0
c["lti"] = np.where(inc_ok, c["loan_amount_num"] / (1000 * c["income_num"]), np.nan)
CUT = V["loan_to_income"]["cut_exact"]
c["lti_band"] = bands(c["lti"], V["loan_to_income"]["bands"], CUT)

c["usable"] = c["decisioned"] & c["dti_ordinal"].notna() & c["cltv_valid"] & inc_ok

# LMI
tr = to_num(c["tract_to_msa_income_percentage"], "tract")
tr = np.where(tr == 0, np.nan, tr) if V["lmi_tract"]["zero_means_missing"] else tr
c["tract_ratio"] = tr
c["lmi"] = pd.array(np.where(np.isnan(tr), None, tr < V["lmi_tract"]["lmi_if_below"]), dtype="boolean")

# Giá
RSV = V["rate_spread"]["valid_range"]
c["rs_num"] = to_num(c["rate_spread"], "rate_spread")
c["rs_valid"] = (c["rs_num"] >= RSV["ge"]) & (c["rs_num"] <= RSV["le"])
c["rs_out_of_range"] = c["rs_num"].notna() & ~c["rs_valid"]
c["pricing"] = c["orig"] & c["rs_valid"]
c["term_num"] = to_num(c["loan_term"], "loan_term")   # chỉ dùng cho bảng thiếu dữ liệu theo kết quả (mục 11)

# Địa lý
st = c["state_code"].astype(str)
c["in_universe"] = st.isin(UNIVERSE).to_numpy()
c["state_missing"] = st.isin(G["missing_values"]).to_numpy()
cty = c["county_code"].astype(str)
c["county_valid"] = cty.str.fullmatch(G["county_code_format"].strip("^$")).fillna(False).to_numpy()
c["msa"] = c["derived_msa_md"].astype(str)

# Lý do từ chối: mã số nguyên, rỗng/NA -> 0
for i in range(1, 5):
    c[f"dr{i}"] = np.nan_to_num(to_num(c[f"denial_reason_{i}"], f"denial_reason_{i}"), nan=0).astype("int16")   # int16: mã Exempt 1111 không vừa int8

print("chuỗi lạ khi chuyển số:", STRANGE or "không có")
print(f"tạo biến xong | {time.time() - t0:.0f}s")

chuỗi lạ khi chuyển số: không có
tạo biến xong | 13s


---
# Giữ bảng Pandas

In [5]:
# giữ bảng Pandas, giải phóng phần còn lại trước khi khởi động Spark
import gc

PD_CORE = c.set_index("row_id").sort_index()
PD_WORK, PD_RES = WORK, RES
for _n in ("c", "raw", "b"):
    globals().pop(_n, None)
gc.collect()
print(f"bảng Pandas: {len(PD_CORE):,} dòng, {PD_CORE.shape[1] + 1} cột")

bảng Pandas: 5,775,325 dòng, 48 cột


---
# Phần 1–2 của `spark_pipeline.py`

# HMDA 2024: pipeline PySpark (rules.yaml 1.1)

Notebook này thay cho notebook PySpark cũ, viết lại theo `rules.yaml` 1.1 và bổ sung các phần còn thiếu
(mô hình hai chiều, HHI, độ nhạy, đầu ra tương đương, phần dự đoán của Spark).

**Chuẩn bị trên Kaggle**
1. Input gồm dataset `hmda-2024-one-year-raw` (`hmda_2024_raw.parquet`) và `hmda-config` (`rules.yaml` bản 1.1, `expected.yaml`).
   Transmittal Sheet lấy từ dataset `hmda-2024-ffiec-files` (Kaggle nhập từ link FFIEC); thiếu file này thì notebook thử tải khi có Internet.
2. Nếu đã chạy notebook Pandas, thêm output của nó làm Input để phần 11 dùng xác suất out-of-fold của Pandas.
3. Settings: Accelerator = None. Ảnh Kaggle có sẵn PySpark 4.0 và Java 21, không cần Internet. Chạy *Run All*.
4. Kết quả ghi vào `/kaggle/working/results/spark/`; các file `eq_*.csv` và `key_numbers.json` có cùng định dạng
   với bên Pandas để `check_equivalence.py` so sánh.

Nguyên tắc: phần tính trên dữ liệu lớn (lọc, tạo biến, groupBy, join, window, vòng lặp hai chiều, MLlib) chạy bằng
Spark; các thống kê cuối trên bảng nhỏ (vài trăm dòng, ví dụ τ hay Spearman giữa 767 lender) dùng cùng công thức
Python với bên Pandas để hai bên ra đúng một con số.

In [6]:
# ===== 0. Cài đặt và khởi tạo Spark =====
import os, glob, json, time, shutil, math, hashlib, subprocess, sys
import numpy as np
import pandas as pd
import yaml
import matplotlib

try:
    get_ipython  # noqa: F821
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import pyspark  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql import types as T

T_START = time.time()
RUN_DUPLICATES = True     # đếm dòng trùng trên 99 cột bằng xxhash64
RUN_MLLIB = True          # logistic regression MLlib, cross-fitting 5 fold (mục 11)


def find_file(pattern, required=True):
    """Tìm file ở mọi dataset/notebook đã Add Input (hoặc thư mục HMDA_INPUT khi chạy ngoài Kaggle)."""
    roots = [os.environ["HMDA_INPUT"]] if os.environ.get("HMDA_INPUT") else ["/kaggle/input"]
    hits = sorted(h for r_ in roots for h in glob.glob(os.path.join(r_, "**", pattern), recursive=True))
    assert hits or not required, f"Không thấy {pattern} trong {roots}: kiểm tra Add Input"
    return hits[0] if hits else None


RAW_SRC = find_file("hmda_2024_raw.parquet")
RULES_PATH = find_file("rules.yaml")
EXP_PATH = find_file("expected.yaml", required=False)
TS_PATH = find_file("*public_ts*.csv", required=False) or find_file("*public_ts*.zip", required=False)
INP = os.path.dirname(RAW_SRC)
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
RES, FIG = f"{OUTROOT}/results/spark", f"{OUTROOT}/figures/spark"
WORK = os.environ.get("HMDA_WORK", "/tmp/hmda")
for d in (RES, FIG, WORK):
    os.makedirs(d, exist_ok=True)
R = yaml.safe_load(open(RULES_PATH, encoding="utf-8"))
assert float(R["version"]) >= 1.1, f"Cần rules.yaml 1.1 trở lên, đang là {R.get('version')}"
if TS_PATH is None and R["source"]["transmittal_sheet"].get("url"):   # chưa có TS trong Input: thử tải từ FFIEC (cần Internet)
    try:
        import urllib.request
        os.makedirs(os.environ.get("HMDA_WORK", "/tmp/hmda"), exist_ok=True)
        TS_PATH = os.path.join(os.environ.get("HMDA_WORK", "/tmp/hmda"), "ts_one_year.zip")
        urllib.request.urlretrieve(R["source"]["transmittal_sheet"]["url"], TS_PATH)
    except Exception as e_:
        print("không tải được Transmittal Sheet:", repr(e_)[:100]); TS_PATH = None
if TS_PATH and TS_PATH.endswith(".zip"):          # file TS tải từ FFIEC dạng zip: giải nén để Spark đọc
    import zipfile
    os.makedirs(os.environ.get("HMDA_WORK", "/tmp/hmda"), exist_ok=True)
    with zipfile.ZipFile(TS_PATH) as z_:
        nm_ = [n for n in z_.namelist() if n.lower().endswith((".csv", ".txt"))][0]
        TS_PATH = z_.extract(nm_, os.environ.get("HMDA_WORK", "/tmp/hmda"))
EXP = yaml.safe_load(open(EXP_PATH, encoding="utf-8")) if EXP_PATH else {}
RAW = f"{WORK}/hmda_2024_raw.parquet"
if not os.path.exists(RAW) or os.path.getsize(RAW) != os.path.getsize(RAW_SRC):
    shutil.copy(RAW_SRC, RAW)

# Bộ nhớ driver = 60% của min(RAM còn trống lúc khởi tạo, giới hạn cgroup của container; Kaggle giới hạn 30 GiB)
try:
    avail_gb = os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 1e9
except (ValueError, OSError):
    avail_gb = 8
for p_ in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    try:
        v_ = open(p_).read().strip()
        if v_.isdigit():
            avail_gb = min(avail_gb, int(v_) / 1e9)
    except OSError:
        pass
driver_gb = max(2, int(avail_gb * 0.6))
t0 = time.time()
spark = (SparkSession.builder.master("local[*]").appName("hmda-v1")
         .config("spark.driver.memory", f"{driver_gb}g")
         .config("spark.sql.shuffle.partitions", str(max(8, 2 * (os.cpu_count() or 2))))
         .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")
         .config("spark.sql.ansi.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir(f"{WORK}/checkpoints")
SPARK_INIT_S = time.time() - t0
print(f"Spark {spark.version} | khởi tạo {SPARK_INIT_S:.1f}s | driver {driver_gb} GB | input {INP}")

V, TH, G = R["variables"], R["thresholds"], R["geography"]
LT_NAME, LP_NAME = R["codes"]["loan_type"], R["codes"]["loan_purpose"]
UNIVERSE = list(G["universe_states"])
KEY = {"rules_version": str(R["version"]), "engine": "spark", "spark_init_s": SPARK_INIT_S}
import platform
KEY["env_spark_notebook"] = (f"python {platform.python_version()}; pyspark {spark.version}; numpy {np.__version__}; "
                             f"pandas {pd.__version__}; java {spark.sparkContext._jvm.System.getProperty('java.version')}")
print(KEY["env_spark_notebook"])
CHECKS = []


def exp_value(path):
    node = EXP
    for p in path.split("."):
        if not isinstance(node, dict) or p not in node:
            return None, None, None
        node = node[p]
    return (node.get("value"), node.get("status"), node.get("tol", 1e-9)) if isinstance(node, dict) else (node, None, 1e-9)


def check(label, value, path=None):
    ev, st, tol = exp_value(path) if path else (None, None, None)
    # tol là sai số TƯƠNG ĐỐI (như ghi trong expected.yaml); kỳ vọng bằng 0 thì so tuyệt đối
    ok = None if (ev is None or value is None or st == "pending") else (
        abs(float(value) - float(ev)) <= tol * (abs(float(ev)) if float(ev) != 0 else 1.0))
    CHECKS.append({"check": label, "value": value, "expected": ev, "expected_status": st, "match": ok})
    flag = "" if ok is None else ("  OK" if ok else "  <-- LỆCH")
    print(f"  {label:<44} {value!s:>16}   kỳ vọng {ev!s:>12} ({st}){flag}")


def save(pdf, name):
    pdf.to_csv(f"{RES}/{name}.csv", index=False)


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.divide(k, n, out=np.full_like(k, np.nan), where=n > 0)
    den = 1 + z ** 2 / n
    ctr = (p + z ** 2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return ctr - half, ctr + half


def spearman(x, y):
    x, y = pd.Series(x).reset_index(drop=True), pd.Series(y).reset_index(drop=True)
    m = x.notna() & y.notna()
    return float(x[m].rank(method="average").corr(y[m].rank(method="average")))


def occupancy_table(occ):
    """Tỷ lệ hồ sơ không phải nơi ở chính theo loan_type, trên các hồ sơ đã qua mọi bước lọc trước bước nơi ở chính."""
    pr_ = next(f["equals"] for f in R["samples"]["core_filters"] if f["step"] == "principal_residence")
    occ = occ.assign(loan_type=occ["loan_type"].astype(str), occupancy_type=occ["occupancy_type"].astype(str))
    t = occ.groupby("loan_type")["n"].sum().rename("n_before_filter").to_frame()
    t["n_non_principal"] = (occ[occ["occupancy_type"] != pr_].groupby("loan_type")["n"].sum()
                            .reindex(t.index).fillna(0).astype("int64"))
    t["share_non_principal"] = t["n_non_principal"] / t["n_before_filter"]
    t.insert(0, "type", [LT_NAME.get(k, k) for k in t.index])
    return t.reset_index().sort_values("loan_type").reset_index(drop=True)


def show():
    if IN_NOTEBOOK:
        plt.show()
    plt.close("all")


def mapper(d):
    return F.create_map(*[F.lit(x) for kv in d.items() for x in kv])

không tải được Transmittal Sheet: URLError(gaierror(-3, 'Temporary failure in name resolution'))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 12:23:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.4 | khởi tạo 11.2s | driver 13 GB | input /kaggle/input/notebooks/study665/hmda-v2-convert
python 3.13.18; pyspark 4.0.4; numpy 2.1.3; pandas 2.3.3; java 21.0.12.1


## 1. Đọc dữ liệu thô, bảng chất lượng dữ liệu, lọc mẫu core
Số dòng sau 7 bước lọc tính trong **một** lần quét bằng các cờ tích lũy (không đếm lại từng bước).

In [7]:
t0 = time.time()
raw = spark.read.parquet(RAW)
ALLCOLS = [c for c in raw.columns if c != "row_id"]
FILT = R["samples"]["core_filters"]
DQ = R["data_quality"]["special_code_table"]

conds, acc = [], F.lit(True)
for f in FILT:
    col = F.col(f["column"])
    if f["step"] == "closed_end":
        open_end_exempt_expr = F.sum((acc & (col == "1111")).cast("long")).alias("open_end_exempt_removed")
    if f["step"] == "principal_residence":
        before_occ = acc                      # hồ sơ qua mọi bước lọc trước bước nơi ở chính
    acc = acc & (col.isin(f["in"]) if "in" in f else (col == f["equals"]))
    conds.append(F.sum(acc.cast("long")).alias(f["step"]))
sc_exprs = [F.sum((F.col(f) == code).cast("long")).alias(f"{f}||{code}") for f in DQ["fields"] for code in DQ["codes"]]
row = raw.agg(F.count(F.lit(1)).alias("n_all"), *conds, open_end_exempt_expr, *sc_exprs).first().asDict()
n_all = row["n_all"]
step_counts = {f["step"]: row[f["step"]] for f in FILT}
open_end_exempt_removed = row["open_end_exempt_removed"]
FP = R["data_quality"]["field_profile"]
fp_exprs = ([F.sum((F.col(f) == code).cast("long")).alias(f"{f}||{code}") for f in ALLCOLS for code in FP["count_codes"]] +
            [F.sum(F.col(f).rlike(FP["numeric_pattern"]).cast("long")).alias(f"{f}||numeric") for f in ALLCOLS])
fp_row = raw.agg(*fp_exprs).first().asDict()
fprof = pd.DataFrame([{"field": f, **{(c_ if c_ else "(empty)"): int(fp_row[f"{f}||{c_}"]) for c_ in FP["count_codes"] + ["numeric"]}}
                      for f in ALLCOLS])
fprof["n"] = n_all
for c_ in [x if x else "(empty)" for x in FP["count_codes"]] + ["numeric"]:
    fprof[f"share_{c_}"] = fprof[c_] / n_all
dq = pd.DataFrame([{"field": f, "code": (code if code else "(empty)"), "count": int(row[f"{f}||{code}"])}
                   for f in DQ["fields"] for code in DQ["codes"]])
act_counts = {r["action_taken"]: r["count"] for r in raw.groupBy("action_taken").count().collect()}
n_dup = 0
if RUN_DUPLICATES:
    n_dup = int(raw.select(F.xxhash64(*ALLCOLS).alias("h")).groupBy("h").count()
                .agg(F.sum(F.col("count") - 1)).first()[0] or 0)
occ = (raw.filter(before_occ).groupBy("loan_type", "occupancy_type").count().toPandas()
       .rename(columns={"count": "n"}))
core = raw.filter(acc)
print(f"đọc {n_all:,} dòng, core {step_counts[FILT[-1]['step']]:,} | trùng {n_dup:,} | {time.time() - t0:.0f}s")

đọc 12,259,199 dòng, core 5,775,325 | trùng 26,755 | 759s


## 2. Tạo biến (rules mục 6) và ghi bảng sạch phân vùng theo `state_code`
`try_cast` biến `NA`, `Exempt`, chuỗi rỗng thành null; NaN (nếu có chuỗi "NaN") cũng đổi thành null để so sánh
không bị lệch (Spark coi NaN lớn hơn mọi số).

In [8]:
t0 = time.time()


def to_num(name):
    x = F.expr(f"try_cast(`{name}` as double)")
    return F.when(F.isnan(x), F.lit(None).cast("double")).otherwise(x)


def band(col, spec, cuts=None):
    rs = lambda v: cuts[v] if isinstance(v, str) else float(v)
    expr = None
    for b in spec:
        lo, hi = rs(b["lo"]), rs(b["hi"])
        cond = F.lit(True)
        if lo != -math.inf:
            cond = cond & (col > F.lit(lo))
        if hi != math.inf:
            cond = cond & (col <= F.lit(hi))
        expr = F.when(cond, F.lit(b["label"])) if expr is None else expr.when(cond, F.lit(b["label"]))
    return expr


D, L = V["debt_to_income_ratio"], V["combined_loan_to_value_ratio"]
CUT = V["loan_to_income"]["cut_exact"]
RSV = V["rate_spread"]["valid_range"]
S_ = R["sampling"]
dti_map = mapper({str(k): int(v) for k, v in D["ordinal_map"].items()})
act = F.col("action_taken")
inc, amt = to_num("income"), to_num("loan_amount")
cltv = to_num("combined_loan_to_value_ratio")
tract = to_num("tract_to_msa_income_percentage")

feat = (core
  .withColumn("decisioned", act.isin(R["samples"]["decisioned"]["action_taken_in"]))
  .withColumn("denied", (act == "3"))
  .withColumn("orig", (act == "1"))
  .withColumn("y", (act == "3").cast("double"))
  .withColumn("dti_ordinal", dti_map[F.regexp_replace(F.col("debt_to_income_ratio"), "[% ]", "")].cast("double"))
  .withColumn("cltv", cltv)
  .withColumn("cltv_valid", F.coalesce((F.col("cltv") > L["valid_range"]["gt"]) & (F.col("cltv") <= L["valid_range"]["le"]), F.lit(False)))
  .withColumn("income_num", inc)
  .withColumn("income_is_na", F.col("income") == "NA")
  .withColumn("loan_amount_num", amt)
  .withColumn("lti", F.when(F.col("income_num") > 0, F.col("loan_amount_num") / (1000 * F.col("income_num"))))
  .withColumn("tract_ratio", F.when(tract != 0, tract) if V["lmi_tract"]["zero_means_missing"] else tract)
  .withColumn("lmi", F.col("tract_ratio") < V["lmi_tract"]["lmi_if_below"])
  .withColumn("rs_num", to_num("rate_spread"))
  .withColumn("rs_valid", F.coalesce((F.col("rs_num") >= RSV["ge"]) & (F.col("rs_num") <= RSV["le"]), F.lit(False)))
  .withColumn("rs_out_of_range", F.col("rs_num").isNotNull() & ~F.col("rs_valid"))
  .withColumn("term_num", to_num("loan_term"))
  .withColumn("in_universe", F.col("state_code").isin(UNIVERSE))
  .withColumn("state_missing", F.col("state_code").isin(G["missing_values"]))
  .withColumn("county_valid", F.coalesce(F.col("county_code").rlike(G["county_code_format"]), F.lit(False)))
  .withColumn("msa", F.col("derived_msa_md"))
  .withColumn("type", mapper(LT_NAME)[F.col("loan_type")])
  .withColumn("purpose", mapper(LP_NAME)[F.col("loan_purpose")])
  .withColumn("sample_bucket", F.pmod(F.pmod(F.col("row_id") * F.lit(S_["hash_multiplier"]).cast("long"),
                                             F.lit(S_["hash_modulus"]).cast("long")), F.lit(S_["buckets"]).cast("long")))
)
for i in range(1, 5):
    feat = feat.withColumn(f"dr{i}", F.coalesce(F.expr(f"try_cast(denial_reason_{i} as int)"), F.lit(0)))
feat = (feat
  .withColumn("dti_band", band(F.col("dti_ordinal"), D["bands"]))
  .withColumn("cltv_band", F.when(F.col("cltv_valid"), band(F.col("cltv"), L["bands"])))
  .withColumn("lti_band", band(F.col("lti"), V["loan_to_income"]["bands"], CUT))
  .withColumn("usable", F.col("decisioned") & F.col("dti_ordinal").isNotNull() & F.col("cltv_valid")
                        & F.coalesce(F.col("income_num") > 0, F.lit(False)))
  .withColumn("pricing", F.col("orig") & F.col("rs_valid"))
)

RAW_KEEP = ["row_id", "lei", "state_code", "county_code", "action_taken", "loan_type", "loan_purpose", "rate_spread",
            "income", "loan_amount"]
KEEPC = RAW_KEEP + [c_ for c_ in feat.columns if c_ not in ALLCOLS and c_ != "row_id"]
CLEAN = f"{WORK}/clean_core"
feat.select(*KEEPC).repartition(R["source"]["partition_clean_by"]).write.mode("overwrite") \
    .partitionBy(R["source"]["partition_clean_by"]).parquet(CLEAN)
c = spark.read.parquet(CLEAN).cache()
n_core = c.count()
print(f"bảng sạch: {n_core:,} dòng, {len(c.columns)} cột | {time.time() - t0:.0f}s")

bảng sạch: 5,775,325 dòng, 43 cột | 295s


## Đối chiếu Parquet do Spark ghi với bảng Pandas, rồi xuất

In [9]:
import json
import shutil
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads

t0 = time.time()
PART = R["source"]["partition_clean_by"]
dset = pads.dataset(CLEAN, format="parquet",
                    partitioning=pads.partitioning(pa.schema([(PART, pa.string())]), flavor="hive"))
spark_cols = [f_.name for f_ in dset.schema]
ids = dset.to_table(columns=["row_id"]).column("row_id").to_numpy()
order = np.argsort(ids, kind="stable")
assert len(ids) == len(PD_CORE) and np.array_equal(ids[order], PD_CORE.index.to_numpy()), "tập row_id khác nhau"


def same(a, b):
    """So hai cột đã xếp theo row_id: chuỗi và bool phải bằng hẳn; số thực sai số tương đối 1e-12; thiếu = thiếu."""
    a, b = pd.Series(a), pd.Series(b)
    na_a, na_b = a.isna().to_numpy(), b.isna().to_numpy()
    if not np.array_equal(na_a, na_b):
        return False, int((na_a != na_b).sum())
    m = ~na_a
    av, bv = a[m].to_numpy(), b[m].to_numpy()
    if np.issubdtype(np.asarray(av).dtype, np.floating) or np.issubdtype(np.asarray(bv).dtype, np.floating):
        av, bv = av.astype(float), bv.astype(float)
        bad = ~np.isclose(av, bv, rtol=1e-12, atol=0)
    else:
        bad = np.asarray(av.astype(str) != bv.astype(str))
    return not bad.any(), int(bad.sum())


rows = []
for col in spark_cols:
    if col == "row_id":
        continue
    sv = dset.to_table(columns=[col]).column(col).to_pandas().to_numpy()[order]
    if col not in PD_CORE.columns:
        rows.append({"column": col, "in_pandas": False, "agree": None, "n_diff": None})
        continue
    ok, nd = same(sv, PD_CORE[col].to_numpy())
    rows.append({"column": col, "in_pandas": True, "agree": ok, "n_diff": nd})
for col in PD_CORE.columns:
    if col not in spark_cols:
        rows.append({"column": col, "in_pandas": True, "in_spark": False, "agree": None, "n_diff": None})
CMP = pd.DataFrame(rows)
CMP["in_spark"] = CMP.get("in_spark", pd.Series(dtype=object)).fillna(True).astype(bool)
both_ = CMP["in_pandas"] & CMP["in_spark"]
print(CMP.to_string(index=False))
n_cmp = int(both_.sum())
assert CMP.loc[both_, "agree"].all(), "Pandas và Spark lệch ở bảng sạch"
print(f"{len(ids):,} dòng x {n_cmp} cột trùng khớp | {time.time() - t0:.0f}s")

                        column  in_pandas agree  n_diff  in_spark
                           lei       True  True     0.0      True
                   county_code       True  True     0.0      True
                  action_taken       True  True     0.0      True
                     loan_type       True  True     0.0      True
                  loan_purpose       True  True     0.0      True
                   rate_spread       True  True     0.0      True
                        income       True  True     0.0      True
                   loan_amount       True  True     0.0      True
                    decisioned       True  True     0.0      True
                        denied       True  True     0.0      True
                          orig       True  True     0.0      True
                             y      False  None     NaN      True
                   dti_ordinal       True  True     0.0      True
                          cltv       True  True     0.0      True
          

/tmp/ipykernel_16/3937622816.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  CMP["in_spark"] = CMP.get("in_spark", pd.Series(dtype=object)).fillna(True).astype(bool)


In [10]:
OUTROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("HMDA_OUT", "output")
DST = os.path.join(OUTROOT, "clean_core")
META = os.path.join(OUTROOT, "results", "clean")
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(CLEAN, DST, ignore=shutil.ignore_patterns("*.crc", "_SUCCESS"))
for d_ in ("results/pandas", "results/spark", "figures"):   # bảng phụ mà phần 1–2 của hai pipeline đã ghi; ở đây không cần
    shutil.rmtree(os.path.join(OUTROOT, d_), ignore_errors=True)
os.makedirs(META, exist_ok=True)

out = pads.dataset(DST, format="parquet", partitioning=pads.partitioning(pa.schema([(PART, pa.string())]), flavor="hive"))
tab_counts = out.to_table(columns=[PART]).column(PART).to_pandas().value_counts().sort_index()
files = [os.path.join(r_, f_) for r_, _, fs in os.walk(DST) for f_ in fs]
manifest = pd.DataFrame({PART: tab_counts.index, "rows": tab_counts.to_numpy()})
manifest.to_csv(f"{META}/clean_core_manifest.csv", index=False)
dic = pd.DataFrame({"column": [f_.name for f_ in out.schema], "type": [str(f_.type) for f_ in out.schema]})
nulls = {c_: int(out.to_table(columns=[c_]).column(c_).null_count) for c_ in dic["column"]}
dic["nulls"] = dic["column"].map(nulls)
dic["source"] = np.where(dic["column"].isin(RAW_KEEP + [PART]), "raw HMDA field, kept as text", "derived (rules.yaml 1.1)")
dic.to_csv(f"{META}/clean_core_columns.csv", index=False)
CMP.to_csv(f"{META}/clean_core_pandas_vs_spark.csv", index=False)
summary = {"rows": int(tab_counts.sum()), "columns": len(dic), "partitions": int(len(tab_counts)),
           "files": len(files), "size_mb": round(sum(os.path.getsize(f_) for f_ in files) / 1e6, 1),
           "columns_compared_with_pandas": n_cmp, "all_agree": bool(CMP.loc[both_, "agree"].all()),
           "rules_version": R["version"]}
json.dump(summary, open(f"{META}/clean_core_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))
assert summary["rows"] == len(PD_CORE)

{
 "rows": 5775325,
 "columns": 43,
 "partitions": 55,
 "files": 55,
 "size_mb": 146.1,
 "columns_compared_with_pandas": 38,
 "all_agree": true,
 "rules_version": "1.1"
}


In [11]:
# In các file kết quả (nhỏ) kèm md5 để lưu lại vào repo
import glob, hashlib, os
for p in sorted(glob.glob("/kaggle/working/results/clean/*")):
    b = open(p, "rb").read()
    print("@@@@ FILE", os.path.relpath(p, "/kaggle/working"), hashlib.md5(b).hexdigest(), len(b))
    print(b.decode("utf-8"))
    print("@@@@ END")

@@@@ FILE results/clean/clean_core_columns.csv 4e9f6fee2aedc8ae23bf8f668e4acfeb 1956
column,type,nulls,source
row_id,int64,0,"raw HMDA field, kept as text"
lei,string,0,"raw HMDA field, kept as text"
county_code,string,0,"raw HMDA field, kept as text"
action_taken,string,0,"raw HMDA field, kept as text"
loan_type,string,0,"raw HMDA field, kept as text"
loan_purpose,string,0,"raw HMDA field, kept as text"
rate_spread,string,0,"raw HMDA field, kept as text"
income,string,0,"raw HMDA field, kept as text"
loan_amount,string,0,"raw HMDA field, kept as text"
decisioned,bool,0,derived (rules.yaml 1.1)
denied,bool,0,derived (rules.yaml 1.1)
orig,bool,0,derived (rules.yaml 1.1)
y,double,0,derived (rules.yaml 1.1)
dti_ordinal,double,1511996,derived (rules.yaml 1.1)
cltv,double,1486063,derived (rules.yaml 1.1)
cltv_valid,bool,0,derived (rules.yaml 1.1)
income_num,double,280709,derived (rules.yaml 1.1)
income_is_na,bool,0,derived (rules.yaml 1.1)
loan_amount_num,double,0,derived (rules.yaml 1.1)
l